# Práctica 2 · Alineamientos de secuencias con Biopython
**Asignatura:** Bioinformática · Grado en Ciencia e Ingeniería de Datos (ULPGC)
**Integrantes:** Lucía Xufang Cruz Toste y Carlota Ayala Pérez

## Ejercicio 1 · Búsqueda en rejilla de penalizaciones de huecos

Se comparan dos secuencias de ADN: una original (`GCATGCATGCATGC`) y otra más corta (`GCATGCGC`). El alineamiento es **global**. La puntuación premia las coincidencias (`+1`), deja las sustituciones en `0` y penaliza la apertura y la extensión de huecos.

Para un tramo de hueco de longitud $L$, la penalización afín que usa Biopython es $g_{apertura}+(L-1)\,g_{extensión}$. Se explora una rejilla de `open_gap` (−1 a −20, paso 1) y `extend_gap` (−0.1 a −5.0, paso 0.1) y se guarda en un DataFrame cuántos alineamientos alcanzan la puntuación óptima. Ese número puede ser mayor que uno cuando hay **empates**, sin que ello implique soluciones de distinta calidad.

In [ ]:
import random
from io import StringIO
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

pd.set_option("display.max_columns", 20)

seq_sana = "GCATGCATGCATGC"
seq_paciente = "GCATGCGC"

# Rejilla elegida dentro del intervalo pedido por el enunciado.
aperturas = np.arange(-1, -21, -1)
extensiones = np.round(np.arange(-0.1, -5.01, -0.1), 2)

alineador = Align.PairwiseAligner()
alineador.mode = "global"
alineador.match_score = 1
alineador.mismatch_score = 0

resultados = []
for apertura in aperturas:
    for extension in extensiones:
        alineador.open_gap_score = float(apertura)
        alineador.extend_gap_score = float(extension)
        alineamientos = alineador.align(seq_sana, seq_paciente)
        resultados.append({
            "open_gap": float(apertura),
            "extend_gap": float(extension),
            "n_optimos": len(alineamientos),
            "mejor_score": alineamientos.score
        })

df_grid = pd.DataFrame(resultados)
print(f"Combinaciones analizadas: {len(df_grid)}")
display(df_grid.head(10))
print("Distribución del número de soluciones óptimas:")
display(df_grid["n_optimos"].value_counts().sort_index().rename_axis("n_optimos").to_frame("combinaciones"))

### Representación del espacio de parámetros

El mapa de calor muestra el **número de óptimos** (no la puntuación). Las filas van de `open_gap = -1` (arriba) a `-20` (abajo) y las columnas de `extend_gap = -0.1` (izquierda) a `-5.0` (derecha). Se ordenan explícitamente la tabla y las etiquetas para que los ejes coincidan con los datos.

In [ ]:
tabla_empates = df_grid.pivot(index="open_gap", columns="extend_gap", values="n_optimos")
# Orden explícito: -1 ... -20 en filas y -0.1 ... -5.0 en columnas.
tabla_empates = tabla_empates.sort_index(ascending=False).sort_index(axis=1, ascending=False)
valores_apertura = tabla_empates.index.to_numpy()
valores_extension = tabla_empates.columns.to_numpy()

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(np.log1p(tabla_empates.values), aspect="auto", origin="upper", cmap="viridis")
pos_x = np.arange(0, len(valores_extension), 5)
ax.set_xticks(pos_x, labels=[f"{valores_extension[i]:.1f}" for i in pos_x])
ax.set_yticks(np.arange(len(valores_apertura)), labels=[f"{v:.0f}" for v in valores_apertura])
ax.set_xlabel("Penalización de extensión (extend_gap)")
ax.set_ylabel("Penalización de apertura (open_gap)")
ax.set_title("Número de alineamientos óptimos según las penalizaciones")
fig.colorbar(im, ax=ax, label="log(1 + nº de óptimos)")
plt.tight_layout()
plt.show()

### Localización matemática de la transición «múltiples empates → solución única»

Con este esquema compiten dos tipos de alineamiento global:

- **Un hueco de seis posiciones**, que conserva 8 coincidencias: $S_A = 8 + o + 5e$.
- **Dos huecos (4 y 2 posiciones)**, que conservan 8 coincidencias: $S_B = 8 + 2o + 4e$.

La igualdad $S_A = S_B$ se da cuando $8+o+5e = 8+2o+4e \Rightarrow \mathbf{o = e}$.
Como $S_A - S_B = e - o$:

- si $o < e$ (la apertura es **más** penalizada que la extensión) gana el hueco único ⇒ **una única solución óptima**;
- si $o = e$ hay empate entre ambas estrategias (y aparecen otras variantes) ⇒ **múltiples óptimos**;
- si $o > e$ (abrir es **más barato** que extender) se reparten los huecos de varias formas ⇒ **también múltiples óptimos**.

Primero se comprueba la frontera en un corte concreto (`extend_gap = -1`) y después se verifica sobre **toda la rejilla**.

In [ ]:
def ejecutar_alineacion(apertura, extension):
    a = Align.PairwiseAligner()
    a.mode = "global"
    a.match_score = 1
    a.mismatch_score = 0
    a.open_gap_score = apertura
    a.extend_gap_score = extension
    return a.align(seq_sana, seq_paciente)

comparacion = []
for apertura in (-0.999, -1.0, -1.001):
    resultado = ejecutar_alineacion(apertura, -1.0)
    comparacion.append({"open_gap": apertura, "extend_gap": -1.0,
                        "n_optimos": len(resultado), "score": resultado.score})
display(pd.DataFrame(comparacion))

print("Solución única justo después de la frontera (open_gap = -1.001, extend_gap = -1):")
print(ejecutar_alineacion(-1.001, -1.0)[0])

o, e = -1.0, -1.0
print(f"En la frontera: S_A = {8 + o + 5*e:.1f}; S_B = {8 + 2*o + 4*e:.1f}")

In [ ]:
# Verificación sobre toda la rejilla: ¿se cumple "única ⇔ open_gap < extend_gap"?
df_grid["regimen"] = np.where(
    np.isclose(df_grid["open_gap"], df_grid["extend_gap"]), "o = e",
    np.where(df_grid["open_gap"] < df_grid["extend_gap"], "o < e", "o > e"))

resumen_regimen = df_grid.groupby("regimen")["n_optimos"].agg(
    combinaciones="size",
    con_solucion_unica=lambda s: int((s == 1).sum()),
    min_optimos="min",
    max_optimos="max")
display(resumen_regimen)

unica = df_grid["n_optimos"] == 1
regla = df_grid["open_gap"] < df_grid["extend_gap"]
print("¿'solución única' coincide con 'open_gap < extend_gap' en las "
      f"{len(df_grid)} combinaciones? ->", bool((unica == regla).all()))

**Interpretación.** El punto de «colapso» no es un único valor, sino la **recta $o = e$** del espacio de parámetros: a un lado (apertura más penalizada que la extensión) el alineador concentra los huecos en un bloque largo y devuelve una única solución; en la recta y al otro lado aparecen empates. La celda anterior comprueba esta regla con las 1000 combinaciones de la rejilla.

**Advertencia (nota importante del enunciado).** Convertir un concepto biológico en un problema de optimización puro es peligroso. En $o = e$ el modelo trata igual **un único evento de inserción/deleción** que **dos eventos** (da la misma puntuación), algo biológicamente poco razonable, porque un indel largo suele ser un solo suceso evolutivo. Además, el «número de óptimos» mide empates del esquema de puntuación, **no** la calidad biológica de los alineamientos: un esquema con más empates no es peor ni mejor por ello. Los valores de penalización deben justificarse con criterios biológicos (p. ej. apertura mucho más penalizada que la extensión), no solo buscando el punto donde el número de soluciones cambia.

In [ ]:
seccion = df_grid[df_grid["extend_gap"].eq(-1.0)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seccion["open_gap"], seccion["n_optimos"], marker="o")
ax.axvline(-1, linestyle="--", color="gray", label="Frontera analítica: o = e = -1")
ax.set_xlabel("Penalización de apertura (extend_gap = -1)")
ax.set_ylabel("Número de alineamientos óptimos")
ax.set_title("Corte del espacio de búsqueda en extend_gap = -1")
ax.legend()
plt.tight_layout()
plt.show()

### Demostración general de la frontera (más allá de «dos estrategias»)

Hasta aquí se compararon dos estrategias concretas. Para asegurar que **ninguna otra configuración** las supera, se razona sobre *todos* los alineamientos globales posibles.

**Notación.** Sea $T$ la secuencia larga (14) y $Q$ la corta (8). Un alineamiento tiene $p$ columnas de emparejamiento, de las cuales $M$ son coincidencias ($M\le p$). Sea $j$ el número de residuos de $Q$ frente a hueco; entonces $T$ tiene $6+j$ residuos frente a hueco y hay $G=6+2j$ columnas de hueco, agrupadas en $r$ tramos. Con $d = o-e$:

$$S \;=\; M + r\,o + (G-r)\,e \;=\; M + r\,d + G\,e \qquad (M\le 8-j)$$

**Caso $o<e$ ($d<0$): solución única.** Se compara cualquier alineamiento con el candidato $8+d+6e$ (un solo bloque, 8 coincidencias):

- Si $j\ge 1$, hay huecos en ambas secuencias y por tanto $r\ge 2$: $S\le (8-j)+2d+(6+2j)e = (8+d+6e) + d + j(2e-1) < 8+d+6e$, porque $d<0$ y $2e-1<0$.
- Si $j=0$ y $r\ge 2$: $S\le 8+2d+6e < 8+d+6e$.
- Si $j=0$, $r=1$ y $M\le 7$: $S\le 7+d+6e < 8+d+6e$.
- Solo queda $j=0$, $r=1$, $M=8$: borrar **un bloque contiguo de 6 posiciones** de $T$ de forma que quede exactamente $Q$. La celda siguiente comprueba que solo la posición de inicio $s=6$ lo consigue ⇒ **el óptimo existe y es único**.

**Caso $o=e$ ($d=0$): empates.** $S = M+Ge \le 8+6e + j(2e-1)$, así que el óptimo exige $j=0$ y $M=8$, y **cualquier** número de tramos $r$ da la misma puntuación. Basta con que existan alineamientos con $M=8$ y $r\ge2$ para tener más de un óptimo (la enumeración siguiente cuenta cuántos).

**Caso $o>e$ ($d>0$): abrir huecos sale más barato que alargarlos.** Ahora cada tramo adicional *suma* $d$, de modo que el bloque único de 6 deja de ser óptimo en cuanto existe un alineamiento con $M=8$ y más tramos. Aquí no hay una fórmula cerrada sencilla, así que se **enumeran exactamente todos los alineamientos** agrupándolos por clase $(M, r, G)$ con programación dinámica. Dentro de cada clase la puntuación es una función lineal de $(o,e)$, por lo que el número de óptimos para cualquier par de penalizaciones se obtiene sumando los contadores de las clases que alcanzan el máximo.

In [ ]:
from collections import defaultdict

def clases_de_alineamiento(s1, s2):
    # Cuenta TODOS los alineamientos globales agrupados por (M, r, G):
    # M = coincidencias, r = tramos de hueco, G = columnas de hueco (sin usar Biopython).
    n, m = len(s1), len(s2)
    tab = [[defaultdict(lambda: defaultdict(int)) for _ in range(m + 1)] for _ in range(n + 1)]
    tab[0][0][None][(0, 0, 0)] = 1
    for i in range(n + 1):
        for j in range(m + 1):
            for ult, d in list(tab[i][j].items()):
                for (M, r, G), c in list(d.items()):
                    if i < n and j < m:
                        tab[i + 1][j + 1]["D"][(M + (s1[i] == s2[j]), r, G)] += c
                    if i < n:
                        tab[i + 1][j]["X"][(M, r + (ult != "X"), G + 1)] += c
                    if j < m:
                        tab[i][j + 1]["Y"][(M, r + (ult != "Y"), G + 1)] += c
    total = defaultdict(int)
    for d in tab[n][m].values():
        for clave, c in d.items():
            total[clave] += c
    return dict(total)

clases = clases_de_alineamiento(seq_sana, seq_paciente)
print(f"Alineamientos globales posibles: {sum(clases.values()):,} agrupados en {len(clases)} clases (M, r, G)")

# 1) Único bloque de 6 huecos que deja exactamente la secuencia corta
inicios = [s for s in range(len(seq_sana) - 5) if seq_sana[:s] + seq_sana[s + 6:] == seq_paciente]
print("Posiciones de inicio válidas para un bloque único de 6 huecos:", inicios)

# 2) Alineamientos con 8 coincidencias y 6 columnas de hueco, según nº de tramos
por_tramos = defaultdict(int)
for (M, r, G), c in clases.items():
    if M == 8 and G == 6:
        por_tramos[r] += c
print("Alineamientos con M=8 y G=6 según nº de tramos r:", dict(sorted(por_tramos.items())))

def n_optimos_exacto(o, e):
    mejor, cuenta = -np.inf, 0
    for (M, r, G), c in clases.items():
        s = M + o * r + e * (G - r)
        if s > mejor + 1e-9:
            mejor, cuenta = s, c
        elif abs(s - mejor) <= 1e-9:
            cuenta += c
    return mejor, cuenta

# 3) El recuento exacto coincide con el de Biopython en las 1000 combinaciones de la rejilla
exacto = np.array([n_optimos_exacto(o, e)[1] for o, e in zip(df_grid["open_gap"], df_grid["extend_gap"])])
print("¿Recuento exacto == recuento de Biopython en toda la rejilla?", bool((exacto == df_grid["n_optimos"].to_numpy()).all()))

# 4) Comprobación sobre valores reales aleatorios (no solo la rejilla)
rng = np.random.default_rng(0)
fallos = 0
for _ in range(20000):
    o, e = -rng.uniform(0.001, 20), -rng.uniform(0.001, 5)
    _, cuenta = n_optimos_exacto(o, e)
    fallos += (cuenta == 1) != (o < e)
print("Fallos de la regla «solución única ⇔ open_gap < extend_gap» en 20000 pares aleatorios:", fallos)

**Lectura de los resultados.** Existe un único bloque de 6 huecos con 8 coincidencias (inicio en 6), mientras que hay 8 alineamientos con 8 coincidencias y dos tramos; en $o=e$ los 9 empatan, que es el valor observado en la frontera. La enumeración exacta reproduce el recuento de Biopython en las 1000 combinaciones y la regla se cumple también en pares de valores reales aleatorios. Con esto, la frontera $o=e$ deja de ser una observación empírica y queda **demostrada para $o<e$ de forma analítica y verificada exhaustivamente para $o\ge e$**.

**Límites de la demostración.** Es específica de este par de secuencias y de las puntuaciones $\text{match}=1$, $\text{mismatch}=0$ con $o,e\le 0$. Con otros pares de secuencias el valor concreto cambia, pero el argumento (comparar $r\,d$ con $G\,e$ y $M$) es el mismo.

## Ejercicio 2 · Comparación «todos contra todos» de proteínas

Se comparan **cinco homólogos reales de la hemoglobina subunidad alfa** (Humano, Ratón, Vaca, Pollo y pez cebra), descargados de UniProt. Con `Bio.SeqIO` se leen las secuencias, se calculan los 25 valores de la matriz mediante alineamiento **global** con **BLOSUM62** y se representa como mapa de calor. Después se repite con otras matrices de sustitución.

**Origen de los datos.** La primera ejecución descarga cada entrada de UniProt (`https://rest.uniprot.org/uniprotkb/<accesión>.fasta`) y guarda `proteinas_reales.fasta`; las siguientes lo reutilizan sin conexión. El código muestra las cabeceras descargadas para poder comprobar que son las proteínas esperadas. Solo si la descarga falla y no existe el archivo, usa un FASTA **sintético** (aviso explícito) para que el cuaderno siga siendo ejecutable.

In [ ]:
import urllib.request

fasta_real = Path("proteinas_reales.fasta")

# Hemoglobina subunidad alfa, entradas de UniProtKB/Swiss-Prot
accesiones = {
    "Humano":    "P69905",
    "Raton":     "P01942",
    "Vaca":      "P01966",
    "Pollo":     "P01994",
    "Pez_cebra": "Q90487",
}

def descargar_fasta_uniprot(accesion):
    url = f"https://rest.uniprot.org/uniprotkb/{accesion}.fasta"
    with urllib.request.urlopen(url, timeout=20) as respuesta:
        return respuesta.read().decode("utf-8")

if not fasta_real.exists():
    try:
        bloques = []
        for especie, acc in accesiones.items():
            texto = descargar_fasta_uniprot(acc)
            cabecera, *lineas = texto.strip().splitlines()
            assert "Hemoglobin subunit alpha" in cabecera, f"{acc}: cabecera inesperada -> {cabecera}"
            bloques.append(f">{especie}|{acc} {cabecera[1:]}\n" + "".join(lineas) + "\n")
        fasta_real.write_text("".join(bloques), encoding="utf-8")
        print("Descarga correcta: se creó", fasta_real)
    except Exception as error:
        print("AVISO: no se pudo descargar de UniProt (", error, ")")

if fasta_real.exists():
    fasta_path = fasta_real
    usando_reales = True
    for linea in fasta_real.read_text(encoding="utf-8").splitlines():
        if linea.startswith(">"):
            print(linea)
else:
    print("AVISO: se usa un FASTA SINTÉTICO; no se pueden sacar conclusiones evolutivas con él.")
    usando_reales = False
    proteina_base = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
    variantes = {
        "Variante_1": {},
        "Variante_2": {8: "S"},
        "Variante_3": {8: "S", 18: "S", 65: "G"},
        "Variante_4": {8: "S", 18: "S", 65: "G", 90: "V", 110: "G", 120: "A"},
        "Variante_5": {8: "S", 18: "S", 65: "G", 90: "V", 110: "G", 120: "A", 25: "D", 45: "E", 80: "I"},
    }

    def mutar_secuencia(original, cambios):
        letras = list(original)
        for posicion, aminoacido in cambios.items():
            letras[posicion] = aminoacido
        return "".join(letras)

    fasta_path = Path("proteinas_practica2_sinteticas.fasta")
    with fasta_path.open("w", encoding="utf-8") as archivo:
        for nombre, cambios in variantes.items():
            archivo.write(f">{nombre} proteina_ejemplo_sintetica\n")
            archivo.write(mutar_secuencia(proteina_base, cambios) + "\n")

registros = list(SeqIO.parse(fasta_path, "fasta"))
nombres = [r.id for r in registros]
n = len(registros)
print("Origen de los datos:", "FASTA real aportado" if usando_reales else "FASTA sintético")
print("Secuencias cargadas:", n)
for r in registros:
    print(r.id, "| longitud:", len(r.seq))

### Matriz de puntuaciones con BLOSUM62

Se calcula la puntuación de cada par (incluida cada secuencia consigo misma) con las mismas penalizaciones de huecos (`-10` apertura, `-0.5` extensión), como en los apuntes.

In [ ]:
def matriz_puntuaciones(registros, nombre_matriz):
    a = Align.PairwiseAligner()
    a.mode = "global"
    a.substitution_matrix = substitution_matrices.load(nombre_matriz)
    a.open_gap_score = -10
    a.extend_gap_score = -0.5
    ids = [r.id for r in registros]
    valores = np.zeros((len(registros), len(registros)))
    for i, ri in enumerate(registros):
        for j, rj in enumerate(registros):
            valores[i, j] = a.score(ri.seq, rj.seq)
    return pd.DataFrame(valores, index=ids, columns=ids)

matriz_blosum62 = matriz_puntuaciones(registros, "BLOSUM62")
print(f"Matriz BLOSUM62 ({n} × {n} = {n*n} puntuaciones)")
display(matriz_blosum62)
assert matriz_blosum62.shape == (n, n)
assert np.allclose(matriz_blosum62.values, matriz_blosum62.values.T)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matriz_blosum62.values, cmap="viridis")
ax.set_xticks(range(n), nombres, rotation=30, ha="right")
ax.set_yticks(range(n), nombres)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{matriz_blosum62.iloc[i, j]:.0f}", ha="center", va="center",
                color="white" if im.norm(matriz_blosum62.iloc[i, j]) < 0.45 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="Puntuación de alineamiento global")
ax.set_title("Similitud entre las proteínas · BLOSUM62")
plt.tight_layout()
plt.show()

### Comparación con otras matrices de sustitución

Se repite el cálculo con **BLOSUM45** y **PAM250**. Cada matriz tiene su propia escala, por lo que **no se comparan los valores absolutos**. Como comparación **exploratoria** se divide cada puntuación por la menor autopuntuación del par:

$$S_{norm}(i,j)=\frac{S(i,j)}{\min\big(S(i,i),\,S(j,j)\big)}$$

**Cautela sobre esta medida.** No es una normalización estándar ni garantiza un índice de similitud acotado o válido para cualquier matriz o par de proteínas: depende de la composición de aminoácidos (la autopuntuación cambia con ella) y de las penalizaciones de huecos. Solo sirve para tener una escala aproximada con la diagonal en 1, y el código comprueba si en estos datos queda por debajo de 1. Para no depender de ella, la comparación principal entre matrices se hace además con medidas **que no dependen de la escala**: el **orden de similitud** y la **correlación de rangos de Spearman** entre las puntuaciones de los pares.

In [ ]:
def normalizar(df):
    diag = np.diag(df.values)
    return pd.DataFrame(df.values / np.minimum.outer(diag, diag), index=df.index, columns=df.columns)

matrices = {"BLOSUM62": matriz_blosum62,
            "BLOSUM45": matriz_puntuaciones(registros, "BLOSUM45"),
            "PAM250": matriz_puntuaciones(registros, "PAM250")}
normalizadas = {k: normalizar(v) for k, v in matrices.items()}

indices_sup = np.triu_indices(n, k=1)
resumen = []
for nombre, df in matrices.items():
    bruto = df.values[indices_sup]
    norm = normalizadas[nombre].values[indices_sup]
    resumen.append({"Matriz": nombre,
                    "Media (bruta)": bruto.mean(), "Mín (bruta)": bruto.min(), "Máx (bruta)": bruto.max(),
                    "Media (normalizada)": norm.mean(), "Mín (normalizada)": norm.min(), "Máx (normalizada)": norm.max()})
display(pd.DataFrame(resumen).round(3))

# Comprobación de la medida exploratoria: ¿queda acotada por 1 fuera de la diagonal en estos datos?
for nombre, df in normalizadas.items():
    fuera = df.values[indices_sup]
    print(f"{nombre:9s} máx. normalizada fuera de la diagonal = {fuera.max():.3f} "
          f"({'<= 1' if fuera.max() <= 1 else 'SUPERA 1: la medida no es una similitud acotada aquí'})")

# Comparación independiente de la escala: correlación de rangos (Spearman) entre matrices
pares = [f"{nombres[i]} - {nombres[j]}" for i, j in zip(*indices_sup)]
tabla_pares = pd.DataFrame({k: v.values[indices_sup] for k, v in matrices.items()}, index=pares)
print("\nCorrelación de rangos de Spearman entre las puntuaciones de los pares:")
display(tabla_pares.rank().corr().round(3))
print("Pares ordenados de más a menos similar (BLOSUM62):")
display(tabla_pares.sort_values("BLOSUM62", ascending=False).round(1))

# ¿Cambia el orden de similitud respecto a la primera secuencia según la matriz?
referencia = nombres[0]
print(f"Orden de similitud (de más a menos parecida a {referencia}):")
for nombre, df in matrices.items():
    orden = df.loc[referencia].drop(referencia).sort_values(ascending=False).index.tolist()
    print(f"  {nombre:9s}", orden)

fig, ax = plt.subplots(figsize=(8, 4))
for nombre, df in normalizadas.items():
    ax.plot(range(1, n), [df.iloc[0, j] for j in range(1, n)], marker="o", label=nombre)
ax.set_xticks(range(1, n), nombres[1:])
ax.set_xlabel(f"Secuencia comparada con {referencia}")
ax.set_ylabel("Puntuación normalizada (1 = idéntica)")
ax.set_title("Efecto de la matriz de sustitución (escala comparable)")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación.** La diagonal compara cada secuencia consigo misma y da las puntuaciones más altas. El mapa de calor es simétrico porque el esquema puntúa igual un par con independencia del orden. Al comparar la **misma proteína** en cinco especies, la puntuación de cada par refleja cuánto se ha conservado la secuencia: lo esperable es que las especies más emparentadas (los mamíferos entre sí) puntúen más alto que las más lejanas (mamíferos frente a pollo y pez cebra). Las tablas anteriores permiten comprobarlo: **hay que contrastar esta expectativa con la salida real**, sin darla por cumplida de antemano.

La correlación de Spearman entre matrices indica si BLOSUM62, BLOSUM45 y PAM250 ordenan los pares de forma parecida, sin depender de su escala. La normalización por la menor autopuntuación se ofrece solo como apoyo visual exploratorio.

**Limitaciones.** Se compara un único gen (hemoglobina alfa) en cinco especies, así que el resultado no sustituye a un análisis filogenético; una puntuación de alineamiento global no es una distancia evolutiva y depende de las penalizaciones de huecos escogidas (`-10` / `-0.5`, las de los apuntes).

**Aportaciones adicionales.** Datos reales de UniProt con comprobación de cabeceras y *fallback* sintético avisado; comprobación automática de simetría y de tamaño; comparación independiente de la escala (orden y Spearman) además de la normalización exploratoria.

## Ejercicio 3 · Escalabilidad de los alineamientos

Se generan pares de secuencias de ADN de **100, 500, 1000, 2000 y 5000** bases con `random` y semilla fija. El tiempo se mide con `time.perf_counter` (alta resolución, en ms) y se reporta la **mediana** de varias repeticiones. Se miden dos operaciones:

- `.score()`: coste de obtener solo la puntuación óptima.
- `.align()` (recuperando el primer alineamiento): coste de **alinear** el par, que incluye el rastreo inverso. No se enumeran todos los óptimos (`len()` sobre el resultado) porque su número puede ser enorme.

In [ ]:
random.seed(42)
longitudes = [100, 500, 1000, 2000, 5000]

def adn_aleatorio(longitud):
    return "".join(random.choices("ACGT", k=longitud))

alineador_tiempo = Align.PairwiseAligner()
alineador_tiempo.mode = "global"
alineador_tiempo.match_score = 1
alineador_tiempo.mismatch_score = -1
alineador_tiempo.open_gap_score = -2
alineador_tiempo.extend_gap_score = -0.5

tiempos = []
for longitud in longitudes:
    a, b = adn_aleatorio(longitud), adn_aleatorio(longitud)

    t_score = []
    for _ in range(3):
        inicio = perf_counter()
        puntuacion = alineador_tiempo.score(a, b)
        t_score.append((perf_counter() - inicio) * 1000)

    t_align = []
    for _ in range(3):
        inicio = perf_counter()
        primer_alineamiento = alineador_tiempo.align(a, b)[0]
        t_align.append((perf_counter() - inicio) * 1000)

    tiempos.append({"longitud": longitud,
                    "score_mediana_ms": np.median(t_score),
                    "score_min_ms": np.min(t_score),
                    "score_max_ms": np.max(t_score),
                    "align_mediana_ms": np.median(t_align),
                    "align_min_ms": np.min(t_align),
                    "align_max_ms": np.max(t_align),
                    "score": puntuacion})

df_tiempos = pd.DataFrame(tiempos)
display(df_tiempos.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for columna, etiqueta in (("score_mediana_ms", ".score()"), ("align_mediana_ms", ".align() (1.er alineamiento)")):
    axes[0].plot(df_tiempos["longitud"], df_tiempos[columna], marker="o", linewidth=2, label=etiqueta)
    axes[1].loglog(df_tiempos["longitud"], df_tiempos[columna], marker="o", linewidth=2, label=etiqueta)

axes[0].set_xlabel("Longitud de cada secuencia (bases)")
axes[0].set_ylabel("Tiempo mediano (ms)")
axes[0].set_title("Escala lineal")
axes[1].set_xlabel("Longitud (bases, escala log)")
axes[1].set_ylabel("Tiempo mediano (ms, escala log)")
axes[1].set_title("Escala log-log (pendiente ≈ exponente)")
for ax in axes:
    ax.grid(alpha=0.3, which="both")
    ax.legend()
plt.tight_layout()
plt.show()

# Exponente empírico: pendiente del ajuste lineal en log-log.
logs_l = np.log(df_tiempos["longitud"].to_numpy(dtype=float))
pendientes = {}
for columna in ("score_mediana_ms", "align_mediana_ms"):
    pendientes[columna] = np.polyfit(logs_l, np.log(df_tiempos[columna].to_numpy()), 1)[0]
    print(f"Exponente empírico de {columna}: {pendientes[columna]:.2f}  (cuadrático teórico = 2)")

factor_long = df_tiempos["longitud"].iloc[-1] / df_tiempos["longitud"].iloc[0]
factor_tiempo = df_tiempos["score_mediana_ms"].iloc[-1] / df_tiempos["score_mediana_ms"].iloc[0]
print(f"Longitud ×{factor_long:.0f} -> tiempo de .score() ×{factor_tiempo:.0f} "
      f"(si fuera exactamente cuadrático: ×{factor_long**2:.0f})")

### Extrapolación a volúmenes grandes

El enunciado pregunta si el *pipeline* escalará de 100 a 1 millón de datos. Con el exponente empírico se estiman dos escenarios, **solo como orden de magnitud** (la extrapolación supone que el comportamiento se mantiene y no considera la memoria ni la caché).

In [ ]:
exp = pendientes["align_mediana_ms"]
t_ref = df_tiempos["align_mediana_ms"].iloc[-1]       # ms para 5000 bases
l_ref = df_tiempos["longitud"].iloc[-1]

# Escenario A: un único alineamiento de dos secuencias de 1 millón de bases.
t_a_ms = t_ref * (1_000_000 / l_ref) ** exp
print(f"A) Un alineamiento de 1.000.000 x 1.000.000 bases (.align()): ~{t_a_ms/1000/3600:,.1f} horas")
print(f"   Celdas de la matriz de programación dinámica: {1_000_000 * 1_000_000:.1e} (O(N·M) en memoria si se guarda el rastreo)")

# Escenario B: un millón de pares de secuencias cortas (100 bases).
t_b_ms = df_tiempos["align_mediana_ms"].iloc[0] * 1_000_000
print(f"B) 1.000.000 de pares de 100 bases, uno tras otro: ~{t_b_ms/1000/3600:,.1f} horas (un solo núcleo)")

**Interpretación.** Al duplicar la longitud, el tiempo se multiplica aproximadamente por cuatro: la matriz de programación dinámica para secuencias de longitudes $N$ y $M$ tiene $N\times M$ celdas, así que para secuencias igual de largas el coste es $O(N^2)$ y el exponente empírico del ajuste log-log queda próximo a 2. Las variaciones respecto a 2 se deben al ordenador, a la caché y a la sobrecarga fija de las secuencias cortas. `.align()` cuesta algo más que `.score()` porque además realiza el rastreo inverso y construye el alineamiento.

**Consecuencia para un ingeniero de datos.** Crecer en *número* de pares escala linealmente (escenario B: viable, paralelizable), pero crecer en *longitud* escala cuadráticamente (escenario A: inviable con programación dinámica exacta). Para volúmenes grandes se recurre a métodos heurísticos (BLAST, FASTA), a paralelismo o a alineamientos con banda.

**Aportaciones adicionales.** Semilla reproducible; mediana de varias repeticiones; medición separada de `.score()` y `.align()`; ajuste log-log del exponente; extrapolación explícita.

## Conclusiones generales

1. Los parámetros de apertura y extensión de huecos influyen en la puntuación y en el número de alineamientos óptimos. La transición entre múltiples empates y solución única es la recta $o = e$: demostrada analíticamente para $o<e$, y verificada con una enumeración exacta de todos los alineamientos para $o\ge e$ y sobre toda la rejilla. Depende del esquema de puntuación, no solo de las secuencias.
2. La comparación de proteínas homólogas con BLOSUM62 resume la similitud de cada pareja en una matriz simétrica. El resultado depende de la matriz de sustitución y de las penalizaciones; las matrices se comparan mejor con medidas independientes de la escala (orden de los pares, correlación de Spearman) que con la normalización exploratoria por autopuntuación.
3. El coste de un alineamiento exacto crece de forma cuadrática con la longitud, lo que limita su uso a secuencias de tamaño moderado.

**Referencia docente:** *T03 · Bioinformática en laboratorio · Sesión 03: Alineamientos* y *Práctica 2*, material proporcionado para la asignatura.